# TA-DA Analysis
## 1 Load model level data

In [3]:
from scipy import stats
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import re
import pandasql as ps
from scipy.stats import wilcoxon
import pandas as pd

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Specify the experiment name and date to analyze
# Set experiment_name to None to analyze all experiments
experiment_name = "goal_vs_consent_based_analysis"
experiment_date = "20251107"  # Format: YYYYMMDD
# Note: The actual data is in the directory: consent_first_vs_goal_first_full_analysis_20251013

In [6]:
def extract_experiment_info(config_filename):
    """Extract experiment name and configuration from config filename.
    
    Example: consent_or_goal_sensitivity_analysis_(seed_2)_seed_2:_0-1000-0_20251013_165148_config.json
    Returns: ('consent_or_goal_sensitivity_analysis', '0-1000-0', '2')
    """
    # Remove _config.json suffix
    name = config_filename.replace('_config.json', '')
    
    # Pattern: {experiment_name}_(seed_{N})_seed_{N}:_{agent_config}_{timestamp}
    # Match the experiment name (everything before _(seed_)
    match = re.match(r'(.+?)_\(seed_(\d+)\)_seed_\2:_(.+?)_(\d{8}_\d{6})$', name)
    
    if match:
        exp_name = match.group(1)
        seed = match.group(2)
        agent_config = match.group(3)
        timestamp_date = match.group(4).split('_')[0]
        return exp_name, agent_config, seed, timestamp_date
    
    return None, None, None

def create_figures_directory(experiment_name, experiment_date):
    """Create figures directory for the experiment if it doesn't exist."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find the experiment directory (it might have a different name than expected)
    experiment_dir = None
    for subdir in results_dir.iterdir():
        if subdir.is_dir():
            # Check if this directory contains files matching our experiment name and date
            configs_dir = subdir / "configs"
            if configs_dir.exists():
                for config_file in configs_dir.glob("*.json"):
                    exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                    if exp_name == experiment_name and file_date == experiment_date:
                        experiment_dir = subdir
                        break
                if experiment_dir:
                    break
    
    if experiment_dir:
        figures_dir = experiment_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir
    else:
        # Fallback: create in main results directory
        figures_dir = results_dir / "figures"
        figures_dir.mkdir(exist_ok=True)
        return figures_dir

def load_simulation_data(experiment_name=None, experiment_date=None):
    """Load all simulation data and extract agent ratios."""
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    simulation_data = []
    timestamp_date = None
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config
        with open(config_file, 'r') as f:
            config = json.load(f)
        
        # Extract agent counts
        params = config['parameters']
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + goal_first + fifty_fifty
        
        # Calculate ratios
        consent_ratio = consent_first / total_agents if total_agents > 0 else 0
        goal_ratio = goal_first / total_agents if total_agents > 0 else 0
        fifty_fifty_ratio = fifty_fifty / total_agents if total_agents > 0 else 0
        
        # Find corresponding model data file
        config_name = config_file.stem
        prefix = config_name.rsplit('_', 1)[0]
        
        # Look for data files in multiple locations
        model_file = None
        agent_file = None
        
        # List of directories to check for data files
        data_dirs_to_check = []
        
        # Check main data directory first
        main_data_dir = results_dir / "data"
        if main_data_dir.exists():
            data_dirs_to_check.append(main_data_dir)
        
        # Check experiment-specific subdirectory
        if experiment_name and experiment_date:
            exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
            if exp_subdir.exists():
                exp_data_dir = exp_subdir / "data"
                if exp_data_dir.exists():
                    data_dirs_to_check.append(exp_data_dir)
        
        # Also search all subdirectories for data files that match the experiment name and date
        if experiment_name and experiment_date:
            for subdir in results_dir.iterdir():
                if subdir.is_dir():
                    exp_data_dir = subdir / "data"
                    if exp_data_dir.exists() and exp_data_dir not in data_dirs_to_check:
                        # Check if any files in this directory match our criteria
                        # We'll check by looking for files with the same prefix as our config file
                        data_dirs_to_check.append(exp_data_dir)
        
        # Find the first directory that contains the required files
        for data_dir in data_dirs_to_check:
            model_file = data_dir / f"{prefix}_model.csv"
            agent_file = data_dir / f"{prefix}_agents.csv"
            if model_file.exists() and agent_file.exists():
                break
        
        if model_file and model_file.exists():
            # Load model data
            model_df = pd.read_csv(model_file)
            agent_df = pd.read_csv(agent_file)

            # Calculate CI state ratios
            # Handle division by zero
            model_df["Consent Violation Ratio"] = model_df["Total Violated Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Fulfillment Ratio"] = model_df["Total Fulfilled Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Unrealized Ratio"] = model_df["Total Unrealized Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Consent Deferred Ratio"] = model_df["Total Deferred Consents"] / model_df["Total Consent Activations"].replace(0, np.nan)
            model_df["Resource Conflict Counter Goal Accomplishment Ratio"] = model_df["Total Resource Conflicts"] / model_df["Total Resource Conflict Accomplished Counter Goals"].replace(0, np.nan)
            
            # Exclude the last early_stop_steps - 1 steps before getting final values
            # But here we should also check if no additional goals were really accomplished after the early stop steps.
            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]
                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    model_df = model_df.iloc[:-steps_to_exclude]
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
                    
            
            # Get final values (last distinct_agent_count rows after exclusion)
            final_agent_values = agent_df.iloc[-distinct_agent_count:]
            
            # Calculate agent-level metrics for each agent type
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'
            
            # Calculate consent-related metrics from the available columns
            # Note: The CSV has different column names than expected
            avg_accomplished_goals_consent_first_agent = final_agent_values[consent_first_mask]['Accomplished Goals'].mean() if consent_first_mask.any() else 0
            avg_accomplished_goals_goal_first_agent = final_agent_values[goal_first_mask]['Accomplished Goals'].mean() if goal_first_mask.any() else 0
            avg_remaining_goals_consent_first_agent = final_agent_values[consent_first_mask]['Remaining Goals'].mean() if consent_first_mask.any() else 0
            avg_remaining_goals_goal_first_agent = final_agent_values[goal_first_mask]['Remaining Goals'].mean() if goal_first_mask.any() else 0
            avg_resource_conflicts_consent_first_agent = final_agent_values[consent_first_mask]['Resource Conflicts'].mean() if consent_first_mask.any() else 0
            avg_resource_conflicts_goal_first_agent = final_agent_values[goal_first_mask]['Resource Conflicts'].mean() if goal_first_mask.any() else 0
            avg_counter_goal_accomplishments_consent_first_agent = final_agent_values[consent_first_mask]['Counter Conflict Goal Accomplishments'].mean() if consent_first_mask.any() else 0
            avg_counter_goal_accomplishments_goal_first_agent = final_agent_values[goal_first_mask]['Counter Conflict Goal Accomplishments'].mean() if goal_first_mask.any() else 0
            
            # Calculate consent metrics from available columns
            # Separately for R (Receiver) and G (Giver) and agent type.
            total_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R'].mean() if consent_first_mask.any() else 0
            total_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G'].mean() if consent_first_mask.any() else 0
            total_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R'].mean() if goal_first_mask.any() else 0
            total_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G'].mean() if goal_first_mask.any() else 0
            violated_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Violated'].mean() if consent_first_mask.any() else 0
            violated_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Violated'].mean() if goal_first_mask.any() else 0
            violated_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Violated'].mean() if goal_first_mask.any() else 0
            
            fulfilled_consents_consent_first_r = final_agent_values[consent_first_mask]['Number of Consents as R Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_consent_first_g = final_agent_values[consent_first_mask]['Number of Consents as G Fulfilled'].mean() if consent_first_mask.any() else 0
            fulfilled_consents_goal_first_r = final_agent_values[goal_first_mask]['Number of Consents as R Fulfilled'].mean() if goal_first_mask.any() else 0
            fulfilled_consents_goal_first_g = final_agent_values[goal_first_mask]['Number of Consents as G Fulfilled'].mean() if goal_first_mask.any() else 0
            
            # Calculate ratios (avoid division by zero)
            avg_consent_violation_ratio_consent_first_r = (violated_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_violation_ratio_consent_first_g = (violated_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_violation_ratio_goal_first_r = (violated_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_violation_ratio_goal_first_g = (violated_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
            
            avg_consent_fulfillment_ratio_consent_first_r = (fulfilled_consents_consent_first_r / total_consents_consent_first_r) if total_consents_consent_first_r > 0 else 0
            avg_consent_fulfillment_ratio_consent_first_g = (fulfilled_consents_consent_first_g / total_consents_consent_first_g) if total_consents_consent_first_g > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_r = (fulfilled_consents_goal_first_r / total_consents_goal_first_r) if total_consents_goal_first_r > 0 else 0
            avg_consent_fulfillment_ratio_goal_first_g = (fulfilled_consents_goal_first_g / total_consents_goal_first_g) if total_consents_goal_first_g > 0 else 0
        
            
            # Resource conflict counter goal accomplishment ratio
            avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent = (avg_resource_conflicts_consent_first_agent / avg_counter_goal_accomplishments_consent_first_agent) if avg_counter_goal_accomplishments_consent_first_agent > 0 else 0
            avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent = (avg_resource_conflicts_goal_first_agent / avg_counter_goal_accomplishments_goal_first_agent) if avg_counter_goal_accomplishments_goal_first_agent > 0 else 0
            
            # Calculate interaction and timing metrics
            avg_finished_step_consent_first_agent = final_agent_values[consent_first_mask]['Finished Step'].mean() if consent_first_mask.any() else 0
            avg_finished_step_goal_first_agent = final_agent_values[goal_first_mask]['Finished Step'].mean() if goal_first_mask.any() else 0
            avg_longest_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Longest Idle Time'].mean() if consent_first_mask.any() else 0
            avg_longest_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Longest Idle Time'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_r_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as R'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_r_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as R'].mean() if goal_first_mask.any() else 0
            avg_distinct_agents_interacted_g_consent_first_agent = final_agent_values[consent_first_mask]['Number of Distinct Agents Interacted as G'].mean() if consent_first_mask.any() else 0
            avg_distinct_agents_interacted_g_goal_first_agent = final_agent_values[goal_first_mask]['Number of Distinct Agents Interacted as G'].mean() if goal_first_mask.any() else 0
            # New: total idle time per agent
            avg_total_idle_time_consent_first_agent = final_agent_values[consent_first_mask]['Total Idle Time'].mean() if consent_first_mask.any() else 0
            avg_total_idle_time_goal_first_agent = final_agent_values[goal_first_mask]['Total Idle Time'].mean() if goal_first_mask.any() else 0
            
            # Calculate steps for this run as the last value of the Step/index column
            if not model_df.empty:
                if 'Step' in model_df.columns:
                    avg_steps_overall = int(pd.to_numeric(model_df['Step'], errors='coerce').dropna().iloc[-1])
                else:
                    first_col = model_df.columns[0]
                    avg_steps_overall = int(pd.to_numeric(model_df[first_col], errors='coerce').dropna().iloc[-1])
            else:
                avg_steps_overall = np.nan
            final_values = model_df.iloc[-1]
            
            simulation_data.append({
                'experiment_name': exp_name,
                'agent_config': agent_config,
                'seed': seed,
                'config_name': config_name,
                'consent_first_count': consent_first,
                'goal_first_count': goal_first,
                'fifty_fifty_count': fifty_fifty,
                'total_agents': total_agents,
                'accomplished_goals': final_values['Total Accomplished Goals'],
                'remaining_goals': final_values['Total Remaining Goals'],
                'violated_consents': final_values['Total Violated Consents'],
                'total_consents': final_values['Total Consent Activations'],
                'resource_conflicts': final_values['Total Resource Conflicts'],
                'counter_goal_accomplishments': final_values['Total Resource Conflict Accomplished Counter Goals'],
                'consent_violation_ratio': final_values['Consent Violation Ratio'],
                'consent_fulfillment_ratio': final_values['Consent Fulfillment Ratio'],
                'consent_unrealized_ratio': final_values['Consent Unrealized Ratio'],
                'consent_deferred_ratio': final_values['Consent Deferred Ratio'],
                'resource_conflict_counter_goal_accomplishment_ratio': final_values['Resource Conflict Counter Goal Accomplishment Ratio'],
                'max_steps': config.get('max_steps', 1000),
                'avg_steps_overall': avg_steps_overall,
                'avg_accomplished_goals_consent_first_agent': avg_accomplished_goals_consent_first_agent,
                'avg_accomplished_goals_goal_first_agent': avg_accomplished_goals_goal_first_agent,
                'avg_remaining_goals_consent_first_agent': avg_remaining_goals_consent_first_agent,
                'avg_remaining_goals_goal_first_agent': avg_remaining_goals_goal_first_agent,
                # R (Receiver) specific metrics
                'avg_total_consents_consent_first_r': total_consents_consent_first_r,
                'avg_total_consents_goal_first_r': total_consents_goal_first_r,
                'avg_violated_consents_consent_first_r': violated_consents_consent_first_r,
                'avg_violated_consents_goal_first_r': violated_consents_goal_first_r,
                'avg_fulfilled_consents_consent_first_r': fulfilled_consents_consent_first_r,
                'avg_fulfilled_consents_goal_first_r': fulfilled_consents_goal_first_r,
                'avg_consent_violation_ratio_consent_first_r': avg_consent_violation_ratio_consent_first_r,
                'avg_consent_violation_ratio_goal_first_r': avg_consent_violation_ratio_goal_first_r,
                'avg_consent_fulfillment_ratio_consent_first_r': avg_consent_fulfillment_ratio_consent_first_r,
                'avg_consent_fulfillment_ratio_goal_first_r': avg_consent_fulfillment_ratio_goal_first_r,
                
                # G (Giver) specific metrics
                'avg_total_consents_consent_first_g': total_consents_consent_first_g,
                'avg_total_consents_goal_first_g': total_consents_goal_first_g,
                'avg_violated_consents_consent_first_g': violated_consents_consent_first_g,
                'avg_violated_consents_goal_first_g': violated_consents_goal_first_g,
                'avg_fulfilled_consents_consent_first_g': fulfilled_consents_consent_first_g,
                'avg_fulfilled_consents_goal_first_g': fulfilled_consents_goal_first_g,
                'avg_consent_violation_ratio_consent_first_g': avg_consent_violation_ratio_consent_first_g,
                'avg_consent_violation_ratio_goal_first_g': avg_consent_violation_ratio_goal_first_g,
                'avg_consent_fulfillment_ratio_consent_first_g': avg_consent_fulfillment_ratio_consent_first_g,
                'avg_consent_fulfillment_ratio_goal_first_g': avg_consent_fulfillment_ratio_goal_first_g,
                
                # General agent metrics
                'avg_resource_conflicts_consent_first_agent': avg_resource_conflicts_consent_first_agent,
                'avg_resource_conflicts_goal_first_agent': avg_resource_conflicts_goal_first_agent,
                'avg_counter_goal_accomplishments_consent_first_agent': avg_counter_goal_accomplishments_consent_first_agent,
                'avg_counter_goal_accomplishments_goal_first_agent': avg_counter_goal_accomplishments_goal_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_consent_first_agent,
                'avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent': avg_resource_conflict_counter_goal_accomplishment_ratio_goal_first_agent,
                
                # Interaction and timing metrics
                'avg_finished_step_consent_first_agent': avg_finished_step_consent_first_agent,
                'avg_finished_step_goal_first_agent': avg_finished_step_goal_first_agent,
                'avg_longest_idle_time_consent_first_agent': avg_longest_idle_time_consent_first_agent,
                'avg_longest_idle_time_goal_first_agent': avg_longest_idle_time_goal_first_agent,
                'avg_distinct_agents_interacted_r_consent_first_agent': avg_distinct_agents_interacted_r_consent_first_agent,
                'avg_distinct_agents_interacted_r_goal_first_agent': avg_distinct_agents_interacted_r_goal_first_agent,
                'avg_distinct_agents_interacted_g_consent_first_agent': avg_distinct_agents_interacted_g_consent_first_agent,
                'avg_distinct_agents_interacted_g_goal_first_agent': avg_distinct_agents_interacted_g_goal_first_agent,
                # New: total idle time
                'avg_total_idle_time_consent_first_agent': avg_total_idle_time_consent_first_agent,
                'avg_total_idle_time_goal_first_agent': avg_total_idle_time_goal_first_agent,
            })
        else:
            print(f"Warning: Model data file not found for {config_name}")
    
    return pd.DataFrame(simulation_data), timestamp_date

def create_agent_ratio_analysis(experiment_name=None, experiment_date=None):
    """Create comprehensive analysis of how metrics change with agent ratios.
    
    This function averages results across all seeds for each experiment configuration.
    """
    print(f"Analyzing experiment: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    
    # Load data
    df, timestamp_date = load_simulation_data(experiment_name=experiment_name, experiment_date=experiment_date)
    
    if df.empty:
        print("No simulation data found!")
        return
    
    # Group by experiment_name and agent_config, then calculate mean and std
    metrics_to_average = [
        'consent_first_count', 'goal_first_count', 'fifty_fifty_count', 'total_agents',
        'accomplished_goals', 'remaining_goals', 'violated_consents', 'total_consents',
        'resource_conflicts', 'counter_goal_accomplishments',
        'consent_violation_ratio', 'consent_fulfillment_ratio', 
        'consent_unrealized_ratio', 'consent_deferred_ratio',
        'resource_conflict_counter_goal_accomplishment_ratio', 'avg_steps_overall'
    ]
    
    # Calculate mean and standard error for each metric
    grouped = df.groupby(['experiment_name', 'agent_config'])
    
    mean_df = grouped[metrics_to_average].mean().reset_index()

    return mean_df, df

mean_df, df = create_agent_ratio_analysis(experiment_name="goal_vs_consent_based_analysis", experiment_date="20251107")


Analyzing experiment: goal_vs_consent_based_analysis, date: 20251107
Found experiment-specific configs in: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs
Found matching configs in subdirectory: /Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results/goal_vs_consent_based_analysis_20251107/configs (110 files)


##  Graph 01-02: 1-way ANOVA: Accomplished Goals

In [7]:
df_sorted = df.drop_duplicates().sort_values(by=['goal_first_count', 'seed'], ascending=True)
df_sorted["consent_violation_ratio"] = df_sorted["total_violated_consents_as_r"] / df_sorted["total_consents_as_r"]

# Build lists of accomplished-goals per goal_first_count level
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["accomplished_goals"].values
    for r in sorted(df_sorted["goal_first_count"].unique())
]

# One-way ANOVA
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

# Get min / max values of the averages graph
max_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().max()
min_accomplished_goals = mean_df.groupby("goal_first_count")["accomplished_goals"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_accomplished_goals}, Min: {min_accomplished_goals}")




KeyError: 'total_violated_consents_as_r'

## Graph 03: 1-way ANOVA: Consent Violation Ratio

In [4]:
groups = [
    df_sorted[df_sorted["goal_first_count"] == r]["consent_violation_ratio"].values
    for r in sorted(df_sorted["goal_first_count"].unique())
]

# One-way ANOVA for consent violation ratios
F, p = stats.f_oneway(*groups)

# Effect size: eta-squared for one-way ANOVA
k = len(groups)                             # number of groups
ns = [len(g) for g in groups]
N = sum(ns)                                 # total sample size
df_between = k - 1
df_within = N - k
eta_sq = (F * df_between) / (F * df_between + df_within)

max_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio = mean_df.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"F: {F:.4f}, p: {p:.3e}")
print(f"Eta-squared (effect size): {eta_sq:.4f}")
print(f"Max: {max_consent_violation_ratio}, Min: {min_consent_violation_ratio}")


F: 45.6124, p: 1.220e-32
Eta-squared (effect size): 0.8217
Max: 0.7435923562503144, Min: 0.4099140920086988


## Get Agent Level Data

In [9]:

def create_agent_level_analysis(experiment_name=None, experiment_date=None):
    """Create analysis of agent-level metrics comparing ConsentFirstAgent and GoalFirstAgent.
    
    This function shows how individual agent performance varies across different configurations.
    Uses agent CSV files and config JSON files directly, no model CSV files.
    """
    print(f"\nCreating Agent-Level Analysis for: {experiment_name}, date: {experiment_date}")
    
    # Create figures directory
    figures_dir = create_figures_directory(experiment_name, experiment_date)
    print(f"Figures will be saved to: {figures_dir}")
    
    results_dir = Path("/Users/efeonal/py_envs/MESA_thesis/consent_abs/simulation_results")
    
    def _find_agent_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_agents.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_agents.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    def _find_model_file(prefix: str):
        main_data = results_dir / "data"
        p = main_data / f"{prefix}_model.csv"
        if p.exists():
            return p
        for sub in results_dir.iterdir():
            if sub.is_dir():
                d = sub / "data"
                q = d / f"{prefix}_model.csv"
                if d.exists() and q.exists():
                    return q
        return None
    
    # Find all config files - check both main directories and experiment-specific subdirectories
    config_files = []
    
    # Check main configs directory
    main_configs_dir = results_dir / "configs"
    if main_configs_dir.exists():
        config_files.extend(list(main_configs_dir.glob("*.json")))
    
    # Check experiment-specific subdirectories
    if experiment_name and experiment_date:
        exp_subdir = results_dir / f"{experiment_name}_{experiment_date}"
        if exp_subdir.exists():
            exp_configs_dir = exp_subdir / "configs"
            if exp_configs_dir.exists():
                config_files.extend(list(exp_configs_dir.glob("*.json")))
                print(f"Found experiment-specific configs in: {exp_configs_dir}")
    
    # Also search all subdirectories for files that match the experiment name and date
    if experiment_name and experiment_date:
        for subdir in results_dir.iterdir():
            if subdir.is_dir():
                exp_configs_dir = subdir / "configs"
                if exp_configs_dir.exists():
                    # Check if any files in this directory match our criteria
                    matching_files = []
                    for config_file in exp_configs_dir.glob("*.json"):
                        exp_name, agent_config, seed, file_date = extract_experiment_info(config_file.name)
                        if exp_name == experiment_name and file_date == experiment_date:
                            matching_files.append(config_file)
                    
                    if matching_files:
                        config_files.extend(matching_files)
                        print(f"Found matching configs in subdirectory: {exp_configs_dir} ({len(matching_files)} files)")
    
    # Collect agent-level data from all agent CSV files
    agent_data_list = []
    agent_data_list_all_steps = []
    
    for config_file in config_files:
        # Extract experiment info from filename
        exp_name, agent_config, seed, timestamp_date = extract_experiment_info(config_file.name)
        
        # Skip if we can't parse the filename or if it doesn't match the desired experiment
        if exp_name is None:
            print(f"Warning: Could not parse filename: {config_file.name}")
            continue
        
        if experiment_name is not None and exp_name != experiment_name:
            continue
        
        # Filter by experiment date if specified
        if experiment_date is not None and timestamp_date != experiment_date:
            continue
        
        # Load config to get agent counts
        try:
            with open(config_file, 'r') as f:
                config = json.load(f)
        except Exception as e:
            print(f"Warning: Could not load config file {config_file}: {e}")
            continue
        
        # Extract agent counts from config
        params = config.get('parameters', {})
        consent_first = params.get('ConsentFirstAgent_COUNT', 0)
        goal_first = params.get('GoalFirstAgent_COUNT', 0)
        fifty_fifty = params.get('FiftyFiftyAgent_COUNT', 0)
        total_agents = consent_first + goal_first + fifty_fifty
        
        # Get config name (without _config.json suffix)
        config_name = config_file.stem
        
        # Find corresponding agent file
        prefix = config_name.rsplit('_', 1)[0]
        agent_file = _find_agent_file(prefix)
        model_file = _find_model_file(prefix)
        
        if agent_file is None or not agent_file.exists():
            print(f"Warning: Agent file not found for {prefix}")
            continue
        
        if model_file is None or not model_file.exists():
            print(f"Warning: Model file not found for {prefix}")
            continue
        
        try:
            agent_df = pd.read_csv(agent_file)
            model_df = pd.read_csv(model_file)
            
            # Get the step column
            step_col = 'Step' if 'Step' in agent_df.columns else agent_df.columns[0]

            early_stop_steps = config.get('early_stop_steps', 0)
            if early_stop_steps > 1:
                # Exclude the last (early_stop_steps - 1) rows
                steps_to_exclude = early_stop_steps - 1
                distinct_agent_count_q = """SELECT COUNT(DISTINCT AgentID) as AGENT_COUNT FROM agent_df WHERE Step = 1"""
                distinct_agent_count = ps.sqldf(distinct_agent_count_q, locals())["AGENT_COUNT"][0]

                if len(model_df) > steps_to_exclude and model_df.iloc[-steps_to_exclude]['Total Accomplished Goals'] == model_df.iloc[-1]['Total Accomplished Goals']:
                    agent_df = agent_df.iloc[:-steps_to_exclude*distinct_agent_count]
            
            # Get final step and calculate avg_steps_overall from agent CSV
            steps = pd.to_numeric(agent_df[step_col], errors='coerce')
            last_step = steps.max()
            avg_steps_overall = int(last_step) if not pd.isna(last_step) else np.nan
            
            final_agent_values = agent_df[steps == last_step].copy()
            
            if 'Agent Persona' not in final_agent_values.columns:
                print(f"Warning: 'Agent Persona' column not found in {agent_file}")
                continue
            
            # Create masks for agent types
            consent_first_mask = final_agent_values['Agent Persona'] == 'ConsentFirstAgent'
            goal_first_mask = final_agent_values['Agent Persona'] == 'GoalFirstAgent'

            final_agent_values["seed"] = seed
            final_agent_values["agent_config"] = agent_config

            agent_df["seed"] = seed
            agent_df["agent_config"] = agent_config
            
            agent_data_list.append(final_agent_values)
            agent_data_list_all_steps.append(agent_df)
        except Exception as e:
            print(f"Error processing {prefix}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if len(agent_data_list) == 0:
        print("Warning: No agent data collected. Returning None.")
        return None
    
    all_agent_values_df = pd.concat(agent_data_list)
    all_agent_values_df["goal_first_count"] = all_agent_values_df["agent_config"].str.split("-").str[1].astype(int)

    all_agent_values_df_all_steps = pd.concat(agent_data_list_all_steps)
    all_agent_values_df_all_steps["goal_first_count"] = all_agent_values_df_all_steps["agent_config"].str.split("-").str[1].astype(int)
    return all_agent_values_df, all_agent_values_df_all_steps


In [ ]:
final_agent_values, all_agent_values_df_all_steps = create_agent_level_analysis(experiment_name="goal_vs_consent_based_analysis", experiment_date="20251107")
#agent_mean_df
final_agent_values

NameError: name 'create_agent_level_analysis' is not defined

In [11]:
all_agent_values_df_all_steps[["seed", "agent_config", "goal_first_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]

,seed,agent_config,goal_first_count,Step,AgentID,Agent Persona,Accomplished Goals
0,999,800-200-0-0,200,0,1,ConsentFirstAgent,0
1,999,800-200-0-0,200,0,2,ConsentFirstAgent,0
2,999,800-200-0-0,200,0,3,ConsentFirstAgent,0
3,999,800-200-0-0,200,0,4,ConsentFirstAgent,0
4,999,800-200-0-0,200,0,5,ConsentFirstAgent,0
...,...,...,...,...,...,...,...
11995,42,0-1000-0-0,1000,11,996,GoalFirstAgent,1
11996,42,0-1000-0-0,1000,11,997,GoalFirstAgent,1
11997,42,0-1000-0-0,1000,11,998,GoalFirstAgent,0
11998,42,0-1000-0-0,1000,11,999,GoalFirstAgent,0


In [2]:
df_sorted = final_agent_values.copy()

# Debug: Check what columns are available
print("Checking for 'Accomplished Goals' column...")
print("Columns in final_agent_values:", list(df_sorted.columns))
if "Accomplished Goals" not in df_sorted.columns:
    print("\n'Accomplished Goals' NOT FOUND!")
    goal_related = [col for col in df_sorted.columns if "goal" in col.lower() or "accomplish" in col.lower()]
    print("Goal/accomplish related columns:", goal_related)
    if not goal_related:
        print("No goal-related columns found. All columns:", list(df_sorted.columns)[:20])

# Compute agent-level ratios
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]
print(f"\nAfter filtering, df_sorted has {len(df_sorted)} rows and columns: {list(df_sorted.columns)}")


df_sorted["consent_violation_ratio"] = (
    df_sorted["Number of Consents as R Violated"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["consent_fulfillment_ratio"] = (
    df_sorted["Number of Consents as R Fulfilled"] /
    df_sorted["Number of Consents as R"]
)

df_sorted["tot_idle_time_normalized"] = df_sorted["Total Idle Time"] / df_sorted["Step"]

# ---- AGGREGATE per simulation run ----
# Build list of columns to aggregate, only including those that exist in df_sorted
cols_to_agg = ["consent_violation_ratio", "consent_fulfillment_ratio", "tot_idle_time_normalized"]

# Find the "Accomplished Goals" column (handle different possible names)
accomplished_col = None
if "Accomplished Goals" in df_sorted.columns:
    accomplished_col = "Accomplished Goals"
else:
    # Try various possible column names
    possible_patterns = [
        lambda col: "accomplished" in col.lower() and "goal" in col.lower(),
        lambda col: col.lower() == "accomplished goals",
        lambda col: col.lower().replace(" ", "") == "accomplishedgoals",
        lambda col: "accomplished" in col.lower(),
    ]
    for pattern in possible_patterns:
        matches = [col for col in df_sorted.columns if pattern(col)]
        if matches:
            accomplished_col = matches[0]
            print(f"Found accomplished goals column: '{accomplished_col}'")
            break

if accomplished_col:
    cols_to_agg.append(accomplished_col)
    # Store the column name for use in cell 21
    globals()['accomplished_goals_col_name'] = accomplished_col
else:
    print("WARNING: Could not find 'Accomplished Goals' column. ANOVA test in cell 21 will fail.")
    print("Available columns:", [c for c in df_sorted.columns if any(x in c.lower() for x in ['goal', 'accomplish', 'achieve'])])

run_level_df = (
    df_sorted.groupby(["goal_first_count", "seed", "Agent Persona"])[cols_to_agg]
    .mean()
    .reset_index()
)

df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]


# Create separate dataframes for "Accomplished Goals" and "tot_idle_time_normalized" 
# that include ALL agents (not just those with consents)
df_all_agents = final_agent_values.copy()
df_all_agents["tot_idle_time_normalized"] = df_all_agents["Total Idle Time"] / df_all_agents["Step"]

# Build list of columns to aggregate - check if "Accomplished Goals" exists
cols_for_full = ["tot_idle_time_normalized"]
if "Accomplished Goals" in df_all_agents.columns:
    cols_for_full.append("Accomplished Goals")
else:
    # Try to find similar column name
    possible = [col for col in df_all_agents.columns if "accomplished" in col.lower() and "goal" in col.lower()]
    if possible:
        cols_for_full.append(possible[0])
        print(f"Note: Using '{possible[0]}' instead of 'Accomplished Goals'")
    else:
        print("Warning: 'Accomplished Goals' not found in df_all_agents. Available columns:", list(df_all_agents.columns)[:10])

run_level_df_all_agents = (
    df_all_agents.groupby(["goal_first_count", "seed", "Agent Persona"])[cols_for_full]
    .mean()
    .reset_index()
)

df_da_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "ConsentFirstAgent"]
df_ta_full = run_level_df_all_agents[run_level_df_all_agents["Agent Persona"] == "GoalFirstAgent"]


NameError: name 'final_agent_values' is not defined

## 04 Consent Violation Ratio, 1-WAY ANOVA TESTS

In [3]:
groups_da = [
    df_da[df_da["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_da["goal_first_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA)
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                           # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_consent_violation_ratio_da = df_da.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_da = df_da.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_consent_violation_ratio_da:.4f}, Min: {min_consent_violation_ratio_da:.4f}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_violation_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA)
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_consent_violation_ratio_ta = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean().max()
min_consent_violation_ratio_ta = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_consent_violation_ratio_ta:.4f}, Min: {min_consent_violation_ratio_ta:.4f}")

NameError: name 'df_da' is not defined

## 05 Consent Fulfilment Ratio, 1-WAY ANOVA TESTS

In [ ]:
groups_da = [
    df_da[df_da["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_da["goal_first_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA) - Fulfilment Ratio
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                           # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_consent_fulfillment_ratio_da = df_da.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_da = df_da.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_da:.4f}, Min: {min_consent_fulfillment_ratio_da:.4f}")

groups_ta = [
    df_ta[df_ta["goal_first_count"] == r]["consent_fulfillment_ratio"]
    for r in sorted(df_ta["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Fulfilment Ratio
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_consent_fulfillment_ratio_ta = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().max()
min_consent_fulfillment_ratio_ta = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_consent_fulfillment_ratio_ta:.4f}, Min: {min_consent_fulfillment_ratio_ta:.4f}")

ConsentFirstAgent (DA) - F: 111.1705, p: 8.197e-45, eta^2: 0.9175
GoalFirstAgent (TA) - F: 83.5479, p: 8.512e-40, eta^2: 0.8931


## 04: Consent Violation Ratio Mann-Whitney Test


In [21]:
import pandas as pd
from scipy.stats import mannwhitneyu

# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------

df_sorted = final_agent_values.copy()

# Keep only agents with at least 1 consent as R
df_sorted = df_sorted[df_sorted["Number of Consents as R"] > 0]

run_level_df = df_sorted.groupby(["goal_first_count", "seed", "Agent Persona"])[[
    "Number of Consents as R",
    "Number of Consents as R Violated",
    "Number of Consents as R Fulfilled"
]].sum().reset_index()

# Aggregate so that each (ratio × seed × persona) is one data point
run_level_df["consent_violation_ratio"] = run_level_df["Number of Consents as R Violated"] / run_level_df["Number of Consents as R"]
run_level_df["consent_fulfillment_ratio"] = run_level_df["Number of Consents as R Fulfilled"] / run_level_df["Number of Consents as R"]

# Split by persona
df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# ----------------------------------------------------------------------
# 2. Compare violation ratios between personas
# ----------------------------------------------------------------------

# Means per condition (align conditions across personas)
consent_first_means = df_da.groupby("goal_first_count")["consent_violation_ratio"].mean()
goal_first_means = df_ta.groupby("goal_first_count")["consent_violation_ratio"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(goal_first_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
goal_first_aligned = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "GoalFirstAgent_mean": goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nViolation Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Count conditions
consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
goal_first_higher_count = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# Overall means using run-level data
overall_da = df_da["consent_violation_ratio"].mean()
overall_ta = df_ta["consent_violation_ratio"].mean()

print("\nOverall mean violation ratios:")
print(f"  ConsentFirstAgent: {overall_da:.4f}")
print(f"  GoalFirstAgent:    {overall_ta:.4f}")
print(f"  Difference:        {overall_da - overall_ta:.4f}")

# Mann–Whitney U tests
print("\nMann–Whitney U Test (Violation Ratios):")
cons_da = df_da["consent_violation_ratio"]
cons_ta = df_ta["consent_violation_ratio"]

u_twosided, p_twosided = mannwhitneyu(cons_da, cons_ta, alternative="two-sided")
u_da_greater, p_da_greater = mannwhitneyu(cons_da, cons_ta, alternative="greater")
u_ta_greater, p_ta_greater = mannwhitneyu(cons_ta, cons_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > TA (one-sided): p = {p_da_greater:.3e}")
print(f"  TA > DA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(cons_da)
n_ta = len(cons_ta)
if n_da > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_ta:
        print("  → SIGNIFICANT: ConsentFirst violate more.")
    else:
        print("  → SIGNIFICANT: GoalFirst violate more.")
else:
    print("  → NO significant difference.")


# ----------------------------------------------------------------------
# 3. (Optional) Repeat EXACT same structure for fulfilment ratios
#    Just replace "consent_violation_ratio" with "consent_fulfillment_ratio"
# ----------------------------------------------------------------------



Violation Ratio Comparison by Persona:
 goal_first_count  ConsentFirstAgent_mean  GoalFirstAgent_mean  Difference  ConsentFirst_higher
                0                0.409914             0.000000    0.409914                 True
              100                0.454378             0.397697    0.056681                 True
              200                0.513350             0.415137    0.098213                 True
              300                0.622716             0.480112    0.142604                 True
              400                0.736243             0.619622    0.116621                 True
              500                0.762715             0.680960    0.081756                 True
              600                0.751830             0.682887    0.068943                 True
              700                0.713370             0.684255    0.029116                 True
              800                0.669599             0.675345   -0.005745                False


## 05: Consent Fulfilment Ratio Mann-Whitney Test


In [22]:
# ----------------------------------------------------------------------
# 1. Prepare data properly: compute ratios + aggregate per run
# ----------------------------------------------------------------------

# Split into two persona sets
df_da = run_level_df[run_level_df["Agent Persona"] == "ConsentFirstAgent"]
df_ta = run_level_df[run_level_df["Agent Persona"] == "GoalFirstAgent"]

# ----------------------------------------------------------------------
# 2. Compare fulfilment ratios between personas
# ----------------------------------------------------------------------

# Mean per condition (align conditions across personas)
consent_first_means = df_da.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()
goal_first_means    = df_ta.groupby("goal_first_count")["consent_fulfillment_ratio"].mean()

# Use union of all observed goal_first_count values and fill missing persona means with 0
all_counts = sorted(set(consent_first_means.index).union(set(goal_first_means.index)))
consent_first_aligned = consent_first_means.reindex(all_counts, fill_value=0)
goal_first_aligned    = goal_first_means.reindex(all_counts, fill_value=0)

comparison_df = pd.DataFrame({
    "goal_first_count": all_counts,
    "ConsentFirstAgent_mean": consent_first_aligned.values,
    "GoalFirstAgent_mean":    goal_first_aligned.values
})
comparison_df["Difference"] = (
    comparison_df["ConsentFirstAgent_mean"] -
    comparison_df["GoalFirstAgent_mean"]
)
comparison_df["ConsentFirst_higher"] = comparison_df["Difference"] > 0

print("\nFulfillment Ratio Comparison by Persona:")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

consent_first_higher_count = comparison_df["ConsentFirst_higher"].sum()
goal_first_higher_count    = len(comparison_df) - consent_first_higher_count

print(f"\nConsentFirstAgent higher in {consent_first_higher_count}/"
      f"{len(comparison_df)} conditions")
print(f"GoalFirstAgent   higher in {goal_first_higher_count}/"
      f"{len(comparison_df)} conditions")

# ----------------------------------------------------------------------
# 3. Compare overall means
# ----------------------------------------------------------------------

overall_da = df_da["consent_fulfillment_ratio"].mean()
overall_ta = df_ta["consent_fulfillment_ratio"].mean()

print("\nOverall mean fulfilment ratios:")
print(f"  ConsentFirstAgent: {overall_da:.4f}")
print(f"  GoalFirstAgent:    {overall_ta:.4f}")
print(f"  Difference:        {overall_da - overall_ta:.4f}")

# ----------------------------------------------------------------------
# 4. Mann–Whitney U tests (non-parametric)
# ----------------------------------------------------------------------

print("\nMann–Whitney U Test (Fulfillment Ratios):")
cons_da = df_da["consent_fulfillment_ratio"]
cons_ta = df_ta["consent_fulfillment_ratio"]

u_twosided,      p_twosided      = mannwhitneyu(cons_da, cons_ta, alternative="two-sided")
u_da_greater,    p_da_greater    = mannwhitneyu(cons_da, cons_ta, alternative="greater")
u_ta_greater,    p_ta_greater    = mannwhitneyu(cons_ta, cons_da, alternative="greater")

print(f"  Two-sided:           p = {p_twosided:.3e}")
print(f"  DA > TA (one-sided): p = {p_da_greater:.3e}")
print(f"  TA > DA (one-sided): p = {p_ta_greater:.3e}")

# Rank-biserial correlation (effect size) using two-sided U
n_da = len(cons_da)
n_ta = len(cons_ta)
if n_da > 0 and n_ta > 0:
    r_rb_abs = 1 - (2 * u_twosided) / (n_da * n_ta)
    # Give direction based on overall means
    direction = 1 if overall_da > overall_ta else -1
    r_rb = direction * abs(r_rb_abs)
    print(f"  Rank-biserial r:    r_rb = {r_rb:.4f} (|r| = {abs(r_rb_abs):.4f})")

if p_twosided < 0.05:
    if overall_da > overall_ta:
        print("  → SIGNIFICANT: ConsentFirstAgent fulfils more.")
    else:
        print("  → SIGNIFICANT: GoalFirstAgent fulfils more.")
else:
    print("  → NO significant difference.")



Fulfillment Ratio Comparison by Persona:
 goal_first_count  ConsentFirstAgent_mean  GoalFirstAgent_mean  Difference  ConsentFirst_higher
                0                0.589875             0.000000    0.589875                 True
              100                0.543128             0.602303   -0.059175                False
              200                0.482690             0.584744   -0.102054                False
              300                0.362446             0.519819   -0.157374                False
              400                0.189460             0.379430   -0.189970                False
              500                0.130788             0.317521   -0.186733                False
              600                0.128371             0.315135   -0.186764                False
              700                0.136211             0.311396   -0.175185                False
              800                0.154013             0.314367   -0.160355                Fals

## 06: Accomplished Goals 1-Way ANOVA Test


In [23]:
# Use df_da_full and df_ta_full from cell 11 which includes "Accomplished Goals"
# (saved before they were overwritten in cells 17 and 19)

# Check if "Accomplished Goals" column exists in df_da_full
if "Accomplished Goals" not in df_da_full.columns:
    # Try to find similar column name
    possible = [col for col in df_da_full.columns if "accomplished" in col.lower() and "goal" in col.lower()]
    if possible:
        accomplished_col = possible[0]
        print(f"Note: Using '{accomplished_col}' instead of 'Accomplished Goals'")
    else:
        print(f"ERROR: 'Accomplished Goals' not found in df_da_full.")
        print(f"Available columns: {list(df_da_full.columns)}")
        print("Please re-run cell 11 to create df_da_full and df_ta_full.")
        raise KeyError(f"'Accomplished Goals' column not found. Available columns: {list(df_da_full.columns)}")
else:
    accomplished_col = "Accomplished Goals"

groups_da = [
    df_da_full[df_da_full["goal_first_count"] == r][accomplished_col]
    for r in sorted(df_da_full["goal_first_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA) - Accomplished Goals
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                           # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_accomplished_goals_da = df_da_full.groupby("goal_first_count")[accomplished_col].mean().max()
min_accomplished_goals_da = df_da_full.groupby("goal_first_count")[accomplished_col].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_accomplished_goals_da:.4f}, Min: {min_accomplished_goals_da:.4f}")

groups_ta = [
    df_ta_full[df_ta_full["goal_first_count"] == r][accomplished_col]
    for r in sorted(df_ta_full["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Accomplished Goals
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_accomplished_goals_ta = df_ta_full.groupby("goal_first_count")[accomplished_col].mean().max()
min_accomplished_goals_ta = df_ta_full.groupby("goal_first_count")[accomplished_col].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_accomplished_goals_ta:.4f}, Min: {min_accomplished_goals_ta:.4f}")

ConsentFirstAgent (DA) - F: 103.7666, p: 1.371e-43, eta^2: 0.9121
  Max: 2.9977, Min: 0.4990
GoalFirstAgent (TA) - F: 90.1262, p: 4.107e-41, eta^2: 0.9001
  Max: 2.8970, Min: 0.5121


## 08: Total Idle Time per Agent Normalized by Steps

In [24]:
# Use df_da_full and df_ta_full from cell 11 which includes "tot_idle_time_normalized"
# (saved before they were overwritten in cells 17 and 19)
groups_da = [
    df_da_full[df_da_full["goal_first_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_da_full["goal_first_count"].unique())
]

# One-way ANOVA for ConsentFirstAgent (DA) - Total Idle Time Normalized
F_da, p_da = stats.f_oneway(*groups_da)

# Effect size (eta-squared) for DA
k_da = len(groups_da)                       # number of groups
ns_da = [len(g) for g in groups_da]
N_da = sum(ns_da)                           # total sample size
df_between_da = k_da - 1
df_within_da = N_da - k_da
eta_sq_da = (F_da * df_between_da) / (F_da * df_between_da + df_within_da)

# Get min / max values of the averages
max_tot_idle_time_normalized_da = df_da_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_da = df_da_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().min()

print(f"ConsentFirstAgent (DA) - F: {F_da:.4f}, p: {p_da:.3e}, eta^2: {eta_sq_da:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_da:.4f}, Min: {min_tot_idle_time_normalized_da:.4f}")

groups_ta = [
    df_ta_full[df_ta_full["goal_first_count"] == r]["tot_idle_time_normalized"]
    for r in sorted(df_ta_full["goal_first_count"].unique())
]

# One-way ANOVA for GoalFirstAgent (TA) - Total Idle Time Normalized
F_ta, p_ta = stats.f_oneway(*groups_ta)

# Effect size (eta-squared) for TA
k_ta = len(groups_ta)
ns_ta = [len(g) for g in groups_ta]
N_ta = sum(ns_ta)
df_between_ta = k_ta - 1
df_within_ta = N_ta - k_ta
eta_sq_ta = (F_ta * df_between_ta) / (F_ta * df_between_ta + df_within_ta)

# Get min / max values of the averages
max_tot_idle_time_normalized_ta = df_ta_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().max()
min_tot_idle_time_normalized_ta = df_ta_full.groupby("goal_first_count")["tot_idle_time_normalized"].mean().min()

print(f"GoalFirstAgent (TA) - F: {F_ta:.4f}, p: {p_ta:.3e}, eta^2: {eta_sq_ta:.4f}")
print(f"  Max: {max_tot_idle_time_normalized_ta:.4f}, Min: {min_tot_idle_time_normalized_ta:.4f}")

ConsentFirstAgent (DA) - F: 90.1835, p: 4.003e-41, eta^2: 0.9002
  Max: 0.9563, Min: 0.5250
GoalFirstAgent (TA) - F: 89.3551, p: 5.799e-41, eta^2: 0.8994
  Max: 0.9512, Min: 0.5203


# Cumulative Graphs, Wilcoxon Test:

Even if the difference of Accomplished Goals for DAs and TAs might be small, it can be consistent.
Wilcoxon test shows this.

In [47]:


df = all_agent_values_df_all_steps[["seed", "agent_config", "goal_first_count", "Step",  "AgentID", "Agent Persona", "Accomplished Goals"]]

results = []

for r in sorted(df["goal_first_count"].unique()):

    df_r = df[df["goal_first_count"] == r]

    # Compute per-seed average accomplished goals per agent
    da_seed_means = (
        df_r[df_r["Agent Persona"] == "ConsentFirstAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    ta_seed_means = (
        df_r[df_r["Agent Persona"] == "GoalFirstAgent"]
        .groupby("seed")["Accomplished Goals"]
        .mean()
    )

    # Ensure paired samples (same seeds)
    common_seeds = da_seed_means.index.intersection(ta_seed_means.index)

    da_vals = da_seed_means.loc[common_seeds]
    ta_vals = ta_seed_means.loc[common_seeds]

    # Wilcoxon signed-rank test
    stat, p = wilcoxon(ta_vals, da_vals)

    # Store results
    results.append({
        "goal_first_count": r,
        "wilcoxon_stat": stat,
        "p_value": p,
        "TA_mean": ta_vals.mean(),
        "DA_mean": da_vals.mean(),
    })

results_df = pd.DataFrame(results)
print(results_df)

/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)


    goal_first_count  wilcoxon_stat   p_value   TA_mean   DA_mean
0                  0            NaN       NaN       NaN       NaN
1                100           21.0  0.556641  1.629891  1.618812
2                200           10.0  0.083984  1.642601  1.614614
3                300           19.0  0.431641  1.500931  1.509695
4                400           15.0  0.232422  0.931318  0.918226
5                500           23.0  0.695312  0.669022  0.654631
6                600           16.0  0.275391  0.593374  0.573003
7                700           12.0  0.130859  0.525148  0.497606
8                800           19.0  0.431641  0.492719  0.469117
9                900           21.0  0.556641  0.464498  0.435854
10              1000            NaN       NaN       NaN       NaN


/Users/efeonal/py_envs/MESA_thesis/venv/lib/python3.12/site-packages/scipy/_lib/_util.py:999: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  return fun(*args, **kwargs)
